# 03 — LSTM Fill Level Prediction
**AI-Driven Waste Collection & Route Optimization**

Time-series forecasting of bin fill levels 24 hours ahead using LSTM.
We use scikit-learn + NumPy to simulate the LSTM approach with a sliding window.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

sensor_df = pd.read_csv('../data/sensor_readings.csv', parse_dates=['timestamp'])
print(f'Loaded {len(sensor_df):,} sensor records')

## 1. Feature Engineering for Time-Series Prediction

In [ ]:
def create_features(df, bin_id, look_back=96, horizon=96):
    """Create sliding window features (96 steps = 24hr at 15-min intervals)"""
    sub = df[df['bin_id'] == bin_id].sort_values('timestamp').reset_index(drop=True)
    sub['hour']    = sub['timestamp'].dt.hour
    sub['dow']     = sub['timestamp'].dt.dayofweek
    sub['is_weekend'] = (sub['dow'] >= 5).astype(int)
    sub['sin_hour']= np.sin(2 * np.pi * sub['hour'] / 24)
    sub['cos_hour']= np.cos(2 * np.pi * sub['hour'] / 24)

    # Rolling statistics as proxy for LSTM memory
    sub['roll_mean_4h']  = sub['fill_level_pct'].rolling(16).mean()
    sub['roll_mean_12h'] = sub['fill_level_pct'].rolling(48).mean()
    sub['roll_std_4h']   = sub['fill_level_pct'].rolling(16).std()
    sub['lag_1h']        = sub['fill_level_pct'].shift(4)
    sub['lag_6h']        = sub['fill_level_pct'].shift(24)
    sub['lag_24h']       = sub['fill_level_pct'].shift(96)
    sub['target_24h']    = sub['fill_level_pct'].shift(-horizon)

    sub = sub.dropna()
    feature_cols = ['fill_level_pct','sin_hour','cos_hour','is_weekend',
                    'roll_mean_4h','roll_mean_12h','roll_std_4h','lag_1h','lag_6h','lag_24h']
    return sub[feature_cols], sub['target_24h']

# Use a high-activity commercial bin
commercial_bins = sensor_df[sensor_df['zone_type']=='commercial']['bin_id'].unique()[:5]
X_all, y_all = [], []
for b in commercial_bins:
    X, y = create_features(sensor_df, b)
    X_all.append(X)
    y_all.append(y)

X_all = pd.concat(X_all).reset_index(drop=True)
y_all = pd.concat(y_all).reset_index(drop=True)

split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 2. Train Gradient Boosting (LSTM proxy) Model

In [ ]:
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                   max_depth=4, subsample=0.8, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_pred = np.clip(y_pred, 0, 100)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('='*40)
print('LSTM-proxy Model Performance (24h Forecast)')
print('='*40)
print(f'MAE  : {mae:.2f}%')
print(f'RMSE : {rmse:.2f}%')
print(f'R²   : {r2:.4f}')
print(f'Accuracy (±10%): {np.mean(np.abs(y_test - y_pred) <= 10)*100:.1f}%')

## 3. Visualise Predictions vs Actual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('LSTM Fill Level Prediction Results — 24h Forecast', fontsize=13, fontweight='bold')

n_show = 192
idx = range(n_show)
axes[0,0].plot(idx, y_test.values[:n_show], label='Actual', color='#2196F3', linewidth=1.5)
axes[0,0].plot(idx, y_pred[:n_show], label='Predicted (LSTM proxy)', color='#FF5722',
               linewidth=1.5, linestyle='--')
axes[0,0].axhline(80, color='red', linestyle=':', linewidth=1, label='Threshold 80%')
axes[0,0].set_title('Actual vs Predicted Fill Level (2-day window)')
axes[0,0].set_ylabel('Fill Level (%)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

residuals = y_test.values - y_pred
axes[0,1].scatter(y_pred[:500], residuals[:500], alpha=0.3, s=10, color='#9C27B0')
axes[0,1].axhline(0, color='red', linewidth=1)
axes[0,1].set_title('Residual Plot')
axes[0,1].set_xlabel('Predicted Fill Level (%)')
axes[0,1].set_ylabel('Residual')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].scatter(y_test.values[:500], y_pred[:500], alpha=0.3, s=10, color='#009688')
lim = [0, 100]
axes[1,0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[1,0].set_title(f'Actual vs Predicted (R² = {r2:.3f})')
axes[1,0].set_xlabel('Actual Fill Level (%)')
axes[1,0].set_ylabel('Predicted Fill Level (%)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

feature_imp = pd.Series(model.feature_importances_, index=X_all.columns).sort_values(ascending=True)
axes[1,1].barh(feature_imp.index, feature_imp.values, color='#FF9800', alpha=0.85)
axes[1,1].set_title('Feature Importance')
axes[1,1].set_xlabel('Importance Score')
axes[1,1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/03_lstm_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Model performance saved.')

## 4. High-Priority Bin Alert System

In [ ]:
# Simulate real-time alert generation
latest_readings = sensor_df.groupby('bin_id').last().reset_index()
latest_readings['predicted_24h'] = latest_readings['fill_level_pct'] + np.random.uniform(5, 20, len(latest_readings))
latest_readings['predicted_24h'] = latest_readings['predicted_24h'].clip(0, 100)
latest_readings['alert_level'] = pd.cut(
    latest_readings['predicted_24h'],
    bins=[0, 50, 70, 85, 100],
    labels=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
)

alert_summary = latest_readings['alert_level'].value_counts()
print('\nReal-Time Alert Summary:')
print('-'*30)
for level in ['CRITICAL','HIGH','MEDIUM','LOW']:
    count = alert_summary.get(level, 0)
    bar   = '█' * count
    print(f'{level:10s}: {count:3d}  {bar}')

print('\nTop 5 Bins Requiring Immediate Collection:')
print(latest_readings.nlargest(5,'predicted_24h')[['bin_id','zone_type','fill_level_pct','predicted_24h','alert_level']].to_string())